In [ ]:
!pip install torchmetrics
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 19.5 MB/s eta 0:00:00


In [ ]:
import csv, re, random, math, pandas
import json
from torchmetrics import ConfusionMatrix

from transformers import AdamW
import torch
import torch.nn as nn
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer
from transformers import BertTokenizer
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch.optim import AdamW

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
with open("FindTaskCorpus.csv") as f:
    reader = csv.reader(f)
    lines = list(reader)

In [ ]:
#print(len(lines))
corpus = [] #{trial_id: str, ot: str, o: str, init_action: str,
           # moves: [(actor, utterance, da, pt, ho, eldL, eldOtGiven, eldLGiven, helOtBelief, helLBelief, helOBelief, eldAction, eldSeesO)]}
i = 1
while i < len(lines):
    #print(i)
    trial_id = lines[i][0]
    ot = lines[i+1][0].lower()
    o = lines[i+1][1].lower()
    init_action = lines[i+1][11]
    init_sees = lines[i+1][12]
    #print(i, trial_id, ot, o)
    i += 2
    if not o: #skip to next trial
        while lines[i][0]:
            i += 1
        i += 1
        if i == 2322: #2381
            break
        continue
    moves = []
    while lines[i][0]:
        if lines[i][6]: #has been annotated
            move = lines[i]
            if move[0]=='hel' and move[11]=='': #eld_action not annotated
                move[11]=0
                #print(move)
            if move[0]=='hel' and move[13]=='': #hel_action not annotated
                print('*******************************************************************')
                print(move)
                if lines[i-1][0]=='hel':
                    print(lines[i-1])
                if lines[i+1][0]=='hel':
                    print(lines[i+1])

                if (not lines[i-1][13] =='') or (not lines[i+1][13] ==''):
                    move[13] = lines[i-1][13]
                    if move[13] == '':
                        move[13] = lines[i+1][13]
                if move[13] == '':
                    move[13] = 6 #12
                print(move)

            if not move[12]: #no eldSeesO annotation
                eld_sees = "0"
                if not move[4].startswith("Close"):
                    if moves:
                        eld_sees = moves[-1][12]
                    elif init_sees:
                        eld_sees = init_sees
                move[12] = eld_sees
            moves.append(tuple(move))
        i += 1
    corpus.append({"trial_id": trial_id, "ot": ot, "o": o, "init_action": init_action, "moves": moves})
    i += 1
    #print(i)

*******************************************************************
['hel', '', '', '', 'Open:C#6', '?', '1', '0', '1', '1', '0', '0', '0', '']
['hel', '', '', '', 'Hold:Pot#3', '?', '1', '0', '1', '1', '2', '0', '', '5']
['hel', '', '', '', 'Open:C#6', '?', '1', '0', '1', '1', '0', '0', '0', '5']
*******************************************************************
['hel', '', '', '', 'Close:C#1', 'C#1', '1', '0', '1', '0', '1', '0', '', '']
['hel', '', '', '', 'Close:C#1', 'C#1', '1', '0', '1', '0', '1', '0', '', 6]
*******************************************************************
['hel', '', '', '', 'Close:C#4', 'C#4', '1', '0', '1', '0', '0', '2', '', '']
['hel', '', '', '', 'Close:C#4', 'C#4', '1', '0', '1', '0', '0', '2', '', 6]
*******************************************************************
['hel', 'Good call.', 'State-y', '', '', 'C#2', '1', '1', '1', '1', '1', '0', '', '']
['hel', '', '', '', 'Takeout:Pot#1', 'C#2', '1', '1', '1', '1', '1', '0', '0', '5']
['hel', 'Good cal

In [ ]:
condensed_corpus = []  # combine consecutive moves by same actor if state doesn't change
for trial in corpus:
    moves = (
        []
    )  # moves: [(actor, utterance, da, pt, ho, eldL, eldOtGiven, eldLGiven, helOtBelief, helLBelief, helOBelief, eldAction, eldSeesO)]
    for move in trial["moves"]:
        if len(moves) == 0:
            moves.append(move)
        else:
            prev_move = moves[-1]
            if prev_move[0] == move[0] and move[6:11] == prev_move[6:11]:
                # if move and prev_move both have pt/HO, consider as separate moves
                if (move[3] or move[4]) and (prev_move[3] or prev_move[4]):
                    moves.append(move)
                    continue
                prev_move = moves.pop(-1)
                new_move = []
                # actors should be the same
                new_move.append(move[0])
                # concatenate utterances
                new_move.append(" ".join([prev_move[1], move[1]]).strip())
                # take latest DA unless it's 'Explain'
                new_move.append(
                    prev_move[2]
                    if prev_move[2] and (not move[2] or move[2] == "Explain")
                    else move[2]
                )
                # take latest existing pt/ho
                new_move.append(
                    prev_move[3] if prev_move[3] and not move[3] else move[3]
                )
                new_move.append(
                    prev_move[4] if prev_move[4] and not move[4] else move[4]
                )
                # take latest L, eldAction, eldSeesO
                # states should be the same
                new_move.extend(move[5:])
                new_move = tuple(new_move)
                moves.append(new_move)
            else:
                moves.append(move)
    condensed_corpus.append(
        {
            "trial_id": trial["trial_id"],
            "ot": trial["ot"],
            "o": trial["o"],
            "init_action": trial["init_action"],
            "moves": moves,
        }
    )

In [ ]:
condensed_corpus[0]

{'trial_id': '2010-05-05-001-0.txt',
 'ot': 'pot',
 'o': 'pot#2',
 'init_action': '1',
 'moves': [('eld',
   'Um , so can we start maybe with the ...',
   'Instruct',
   '',
   '',
   '?',
   '0',
   '0',
   '0',
   '0',
   '0',
   '',
   '0',
   ''),
  ('eld',
   'Taking the pot out , um. And then you can help me fill it with water maybe?',
   'Instruct',
   '',
   '',
   '?',
   '1',
   '0',
   '0',
   '0',
   '0',
   '',
   '0',
   ''),
  ('hel',
   'Of course.',
   'Acknowledge',
   '',
   '',
   '?',
   '1',
   '0',
   '1',
   '0',
   '0',
   '0',
   '0',
   '6'),
  ('hel',
   'Okay , so the pots and pans are down here?',
   'Query-yn',
   '',
   'Touch:C#6',
   '?',
   '1',
   '0',
   '1',
   '1',
   '0',
   '5',
   '0',
   '4'),
  ('eld',
   'um , Well , you can ...',
   'Reply-y',
   '',
   '',
   '?',
   '1',
   '1',
   '1',
   '1',
   '0',
   '',
   '0',
   ''),
  ('hel', '', '', '', 'Open:C#6', '?', '1', '1', '1', '1', '0', '6', '2', '5'),
  ('eld',
   'Yes , there are , but

In [ ]:
LOC = {"bed", "c", "corner", "counter", "d", "freezer", "fridge", "sink", "stove", "table"}
PLURAL = {"bowls": "bowl", "dishes": "dish", "glasses": "glass", "grapes": "grape",
          "napkins": "napkin", "plates": "plate", "pots": "pot"}
SINGULAR = {s:p for p,s in PLURAL.items()}
OBJ_SUB = {"bowl": "dish", "plate": "dish", "salad bowl": "bowl",
           "glass": "cup", "grape": "fruit", "macaroni": "pasta",
           "bottle of oil": "bottle", "knife": "silverware",
           "fork": "silverware", "spoon": "silverware"}
OBJ_GROUPS = {"bowl": {"dish", "salad bowl"}, "cup": {"glass"},
              "dish": {"bowl", "plate", "salad bowl"}, "fruit": {"grape"},
              "glass": {"cup"}, "grape": {"fruit"}, "ice cream": {"bowl"},
              "juice": {"cup", "glass"}, "milk": {"cup", "glass"},
              "pasta": {"dish", "plate", "pot"}, "plate": {"dish"},
              "salad": {"bowl", "dish", "plate"}, "salad bowl": {"bowl", "dish"},
              "silverware": {"spoon"}, "spoon": {"silverware"}}

data = []
for trial in condensed_corpus:
    for i,move in enumerate(trial["moves"]):
        if move[0] == "eld": continue

        next_actor = trial["moves"][i+1][0] if i+1 <len(trial["moves"]) else ""
        #if next_actor == "hel" or next_actor == "": continue
        #if next_actor == "": continue

        datapoint = {}
        datapoint["id"] = "{}|{}".format(trial["trial_id"], i+1)
        datapoint["ot"] = trial["ot"]

        utt = move[1].lower() if move[1] else "action"
        utt = re.sub("\s\s+", " ", re.sub("[.?!,]|xxx", "", utt))
        datapoint["utt"] = utt
        datapoint["da"] = move[2]
        datapoint["action"] = int(move[13])
        datapoint["actor"] = move[0]
        datapoint["next_actor"] = next_actor

        if i == 0:
            prev_utt = "start"
        else:
            if trial["moves"][i-1][1]:
                prev_utt = trial["moves"][i-1][1].lower()
            else:
                prev_utt = "action"
        prev_utt = re.sub("\s\s+", " ", re.sub("[.?!,]|xxx", "", prev_utt))
        datapoint["prev_utt"] = prev_utt
        datapoint["prev_da"] = trial["moves"][i-1][2] if i > 0 else ""

        datapoint["prev_actor"] = trial["moves"][i-1][0] if i > 0 else ""
        datapoint["prev_ot_given"] = int(trial["moves"][i-1][6]) if i > 0 else 0
        datapoint["prev_l_given"] = int(trial["moves"][i-1][7]) if i > 0 else 0
        datapoint["prev_hel_ot_belief"] = int(trial["moves"][i-1][8]) if i > 0 else 0
        datapoint["prev_hel_l_belief"] = int(trial["moves"][i-1][9]) if i > 0 else 0
        datapoint["prev_hel_o_belief"] = int(trial["moves"][i-1][10]) if i > 0 else 0

        eld_ot = PLURAL.get(trial["ot"], trial["ot"])
        eld_l = re.sub(' *, *', ',', move[5].lower()).strip('[]').split(',')
        eld_o = trial["o"].lower()
        datapoint["o"] = eld_o

        pt_target = 0
        pt = 0
        if move[3]:
            targets = re.sub(' *, *', ',', move[3].lower()).strip('[]').split(',')
            target_name = targets[0].split('#')[0] #check first target for OBJ or LOC
            if target_name in LOC:
                pt_target = 1
                if eld_l[0] == '?' or targets[0] in eld_l:
                    pt = 1
                else:
                    pt = 2
            else: #target is OBJ
                pt_target = 2
                if eld_o in targets:
                    pt = 1
                else:
                    target_name = PLURAL.get(target_name, target_name)
                    if OBJ_SUB.get(target_name, target_name) == OBJ_SUB.get(eld_ot, eld_ot):
                        pt = 2
                    else:
                        pt = 3
        datapoint["pt_target"] = pt_target
        datapoint["pt"] = pt

        ho_target = 0
        ho = 0
        ho_type = ""
        if move[4]:
            ho_type, targets = move[4].split(':')
            targets = re.sub(' *, *', ',', targets.lower()).strip('[]').split(',')
            target_name = targets[0].split('#')[0] #check first target for OBJ or LOC
            if target_name in LOC:
                ho_target = 1
                if eld_l[0] == '?' or targets[0] in eld_l:
                    ho = 1
                else:
                    ho = 2
            else: #target is OBJ
                ho_target = 2
                if eld_o in targets:
                    ho = 1
                else:
                    target_name = PLURAL.get(target_name, target_name)
                    if OBJ_SUB.get(target_name, target_name) == OBJ_SUB.get(eld_ot, eld_ot):
                        ho = 2
                    else:
                        ho = 3
        datapoint["ho_target"] = ho_target
        datapoint["ho"] = ho
        datapoint["ho_type"] = ho_type

        for j in range(i-1, -1, -1):
            if trial["moves"][j][0] == 'hel':
                prev_eld_action = int(trial["moves"][j][11])
                break
        else:
            prev_eld_action = int(trial["init_action"])
        datapoint["prev_eld_action"] = prev_eld_action

        datapoint["hel_ot_belief"] = int(move[8])
        datapoint["hel_l_belief"] = int(move[9])
        datapoint["hel_o_belief"] = int(move[10])
        #print(move)
        datapoint["eld_action"] = int(move[11])
        if i+1<len(trial['moves']) and trial['moves'][i+1][0]== 'eld':
            datapoint["eld_da"]=trial['moves'][i+1][2]
        #elif i+2<len(trial['moves']) and trial['moves'][i+2][0]== 'eld':
            #datapoint["eld_da"]=trial['moves'][i+2][2]
        #elif i+3<len(trial['moves']) and trial['moves'][i+3][0]== 'eld':
            #datapoint["eld_da"]=trial['moves'][i+3][2]
        else:
            datapoint["eld_da"]=""
        datapoint["eld_sees_o"] = int(move[12])

        data.append(datapoint)

In [ ]:
data[0]

{'id': '2010-05-05-001-0.txt|3',
 'ot': 'pot',
 'utt': 'of course',
 'da': 'Acknowledge',
 'action': 6,
 'actor': 'hel',
 'next_actor': 'hel',
 'prev_utt': 'taking the pot out um and then you can help me fill it with water maybe',
 'prev_da': 'Instruct',
 'prev_actor': 'eld',
 'prev_ot_given': 1,
 'prev_l_given': 0,
 'prev_hel_ot_belief': 0,
 'prev_hel_l_belief': 0,
 'prev_hel_o_belief': 0,
 'o': 'pot#2',
 'pt_target': 0,
 'pt': 0,
 'ho_target': 0,
 'ho': 0,
 'ho_type': '',
 'prev_eld_action': 1,
 'hel_ot_belief': 1,
 'hel_l_belief': 0,
 'hel_o_belief': 0,
 'eld_action': 0,
 'eld_da': '',
 'eld_sees_o': 0}

In [ ]:
for datapoint in data:
    if (datapoint['prev_hel_ot_belief']==1 and datapoint['prev_hel_l_belief']==0
        and datapoint['prev_hel_o_belief']==0 and datapoint['prev_actor']=='eld'):
        print(datapoint)

{'id': '2010-05-05-001-0.txt|8', 'ot': 'pot', 'utt': 'action', 'da': '', 'action': 5, 'actor': 'hel', 'next_actor': 'eld', 'prev_utt': 'yes there are but may be not my the one that i want to use', 'prev_da': 'Reply-n', 'prev_actor': 'eld', 'prev_ot_given': 1, 'prev_l_given': 0, 'prev_hel_ot_belief': 1, 'prev_hel_l_belief': 0, 'prev_hel_o_belief': 0, 'o': 'pot#2', 'pt_target': 0, 'pt': 0, 'ho_target': 1, 'ho': 1, 'ho_type': 'Open', 'prev_eld_action': 6, 'hel_ot_belief': 1, 'hel_l_belief': 1, 'hel_o_belief': 0, 'eld_action': 6, 'eld_da': 'State-n', 'eld_sees_o': 2}
{'id': '2010-05-05-001-0.txt|10', 'ot': 'pot', 'utt': 'this one', 'da': 'Query-yn', 'action': 5, 'actor': 'hel', 'next_actor': 'eld', 'prev_utt': 'no we need a bigger one', 'prev_da': 'State-n', 'prev_actor': 'eld', 'prev_ot_given': 1, 'prev_l_given': 0, 'prev_hel_ot_belief': 1, 'prev_hel_l_belief': 0, 'prev_hel_o_belief': 0, 'o': 'pot#2', 'pt_target': 0, 'pt': 0, 'ho_target': 2, 'ho': 2, 'ho_type': 'Touch', 'prev_eld_action':

In [ ]:
len(data)

670

In [ ]:
random.seed(0)
idxs = list(range(len(data)))
random.shuffle(idxs)
train_len = int(0.7*len(data))
valid_len = int(0.15*len(data))
train_idxs = idxs[:train_len]
valid_idxs = idxs[train_len:train_len+valid_len]
test_idxs = idxs[train_len+valid_len:]

In [ ]:
objs = []
spec_objs = []

for i in train_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'hel' and (re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '3'))):
        #print(dp["ot"])
        if dp["ot"] not in objs:
            objs.append(dp["ot"])
        if dp["o"] not in spec_objs:
            spec_objs.append(dp["o"])


for i in valid_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'hel' and (re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '3'))):
        #print(dp["ot"])
        if dp["ot"] not in objs:
            objs.append(dp["ot"])
        if dp["o"] not in spec_objs:
            spec_objs.append(dp["o"])

for i in test_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'hel' and (re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '3'))):
        #print(dp["ot"])
        if dp["ot"] not in objs:
            objs.append(dp["ot"])
        if dp["o"] not in spec_objs:
            spec_objs.append(dp["o"])

In [ ]:
objs

['pot',
 'spoon',
 'tray',
 'glass',
 'grapes',
 'pasta',
 'salad',
 'plates',
 'silverware',
 'cup',
 'juice',
 'plate',
 'ice cream',
 'napkins',
 'glasses',
 'salad bowl']

In [ ]:
spec_objs

['pot#1',
 'spoon#1',
 'tray#2',
 'plate#1',
 'glass#2',
 'grapes',
 'macaroni#1',
 'pasta#1',
 'salad#1',
 'plates',
 'glass#1',
 'silverware',
 'cup#1',
 'juice#1',
 'ice cream#1',
 'napkins',
 'cup#2',
 'glasses',
 'bowl#1',
 'tray#1',
 'pot#2',
 'spoon#4',
 'plates#2']

In [ ]:
states = {(i,j,k):0 for i in range(3) for j in range(4) for k in range(4)}
for d in [data[i] for i in train_idxs]:
    states[(d["hel_ot_belief"], d["hel_l_belief"], d["hel_o_belief"])] += 1
states

{(0, 0, 0): 7,
 (0, 0, 1): 0,
 (0, 0, 2): 0,
 (0, 0, 3): 0,
 (0, 1, 0): 4,
 (0, 1, 1): 0,
 (0, 1, 2): 0,
 (0, 1, 3): 0,
 (0, 2, 0): 0,
 (0, 2, 1): 0,
 (0, 2, 2): 0,
 (0, 2, 3): 0,
 (0, 3, 0): 0,
 (0, 3, 1): 0,
 (0, 3, 2): 0,
 (0, 3, 3): 0,
 (1, 0, 0): 124,
 (1, 0, 1): 14,
 (1, 0, 2): 8,
 (1, 0, 3): 0,
 (1, 1, 0): 178,
 (1, 1, 1): 84,
 (1, 1, 2): 12,
 (1, 1, 3): 14,
 (1, 2, 0): 12,
 (1, 2, 1): 0,
 (1, 2, 2): 0,
 (1, 2, 3): 0,
 (1, 3, 0): 11,
 (1, 3, 1): 0,
 (1, 3, 2): 0,
 (1, 3, 3): 0,
 (2, 0, 0): 0,
 (2, 0, 1): 0,
 (2, 0, 2): 0,
 (2, 0, 3): 0,
 (2, 1, 0): 0,
 (2, 1, 1): 0,
 (2, 1, 2): 0,
 (2, 1, 3): 0,
 (2, 2, 0): 0,
 (2, 2, 1): 0,
 (2, 2, 2): 0,
 (2, 2, 3): 0,
 (2, 3, 0): 0,
 (2, 3, 1): 0,
 (2, 3, 2): 0,
 (2, 3, 3): 0}

In [ ]:
random.seed(345)
train_aug = []

for i in train_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    # add (0,0,0) and (0,1,0) instances
    # take random HEL actions and replace utterance with a random (0,0,0)/(0,1,0) utterance
    det_ot_utts = ['uh what did you need',
                   'alright what are we looking for',
                   'what are we looking for',
                   'okay what am i looking for',
                   'what do you need to find first']
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'eld' or dp["prev_da"] not in ["State-y", "State-n"]) and
        (dp["prev_actor"] == 'eld' or "Open" in condensed_corpus[trial_idx]["moves"][move_idx-1][4]) and
        (move_idx < 2 or condensed_corpus[trial_idx]["moves"][move_idx-2][10] != '3')): # hel action is not check ot
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        to_add = []
        det_ot_utts_copy = list(det_ot_utts)
        random.shuffle(det_ot_utts_copy)
        for j in range(2): #1
            if ((hel_l_belief and random.random() < 0.15) or
                (not hel_l_belief and random.random() < 0.75)):
                if len(det_ot_utts_copy) == 0: continue
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(len(to_add)+1)
                dp_aug["utt"] = det_ot_utts_copy.pop(0)
                dp_aug["da"] = "Query-w"
                dp_aug["action"] = 1
                dp_aug["hel_ot_belief"] = 0
                dp_aug["hel_l_belief"] = hel_l_belief
                dp_aug["hel_o_belief"] = 0
                dp_aug["pt"] = 0
                dp_aug["pt_target"] = 0
                dp_aug["ho"] = 0
                dp_aug["ho_target"] = 0
                dp_aug["ho_type"] = ''
                dp_aug["eld_action"] = 1
                dp_aug["eld_da"] = "Reply-w"
                dp_aug["next_actor"] = "eld"
                to_add.append(dp_aug)
        train_aug.extend(to_add)



    # add (2,0,0) and (2,1,0) instances
    # take random HEL actions containing the target OT and replace OT in utterance with a random different OT (that isn’t a sub/superclass)
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and dp["prev_actor"] == 'hel'
        and ((re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '2'))):
        #print(dp)
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        x = objs[:]
        x.remove(dp["ot"])
        to_add = []
        for j in range(50): #50, 100, 150
            if ((hel_l_belief and random.random() < 0.02) or
                (not hel_l_belief and random.random() < 0.9)):
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(len(to_add)+1)
                if len(x) >0:
                    #ot_aug = objs[random.randint(0, len(x)-1)]
                    ot_aug = x[random.randint(0, len(x)-1)]
                    x.remove(ot_aug)
                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], ot_aug)
                    dp_aug["da"] = dp["da"]
                    dp_aug["action"] = dp["action"]
                    dp_aug["hel_ot_belief"] = 2
                    dp_aug["hel_l_belief"] = hel_l_belief
                    dp_aug["hel_o_belief"] = 0
                    dp_aug["pt"] = dp["pt"]
                    if dp_aug["pt_target"]==2:
                        dp_aug["pt"] = 2 #dp["pt"]
                    dp_aug["pt_target"] = dp["pt_target"]
                    dp_aug["ho"] = dp["ho"]
                    if dp_aug["ho_target"]==2:
                        dp_aug["ho"] = 2 #dp["pt"]
                    dp_aug["ho_target"] = dp["ho_target"]
                    dp_aug["ho_type"] = dp["ho_type"]
                    dp_aug["eld_action"] = 1
                    dp_aug["eld_da"] = "Instruct"
                    dp_aug["next_actor"] = "eld"
                    if not dp_aug in to_add:
                        to_add.append(dp_aug)
                #print(dp_aug)
        #print(to_add)
        train_aug.extend(to_add)

    ## add (1,1,2) instances
    ## randomly replace O in pointing and HO actions with a different O
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"]==1
        and (not dp["hel_o_belief"] == 0) and ((dp['pt'] and dp['pt_target']==2) or (dp['ho'] and dp['ho_target']==2))):
        #print(dp)
        x = spec_objs[:]
        if dp["o"] in x:
            x.remove(dp["o"])
        to_add=[]
        for j in range(3): #2 #3, 5
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(len(to_add)+1)
            if len(x) >0:
                dp_aug["utt"] = dp["utt"]
                o_aug = x[random.randint(0, len(x)-1)]
                x.remove(o_aug)
                if dp["utt"]:
                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], o_aug)
                    dp_aug["utt"] = dp["utt"].replace(dp["o"], o_aug)
                dp_aug["da"] = dp["da"]
                dp_aug["action"] = dp["action"]
                dp_aug["hel_ot_belief"] = 1
                dp_aug["hel_l_belief"] = 1
                dp_aug["hel_o_belief"] = 2
                dp_aug["pt"] = dp["pt"]
                if dp_aug["pt_target"]==2:
                    dp_aug["pt"] = 3 #dp["pt"]
                dp_aug["pt_target"] = dp["pt_target"]
                dp_aug["ho"] = dp["ho"]
                if dp_aug["ho_target"]==2:
                    dp_aug["ho"] = 3 #dp["pt"]
                dp_aug["ho_target"] = dp["ho_target"]
                dp_aug["ho_type"] = dp["ho_type"]
                dp_aug["eld_action"] = 1 #gives spec ot
                dp_aug["eld_da"] = "Instruct"
                dp_aug["next_actor"] = "eld"
                if not dp_aug in to_add:
                    to_add.append(dp_aug)
              #print(dp_aug)
          #print(to_add)
        train_aug.extend(to_add)

    # add (1,2,0) instances
    # randomly replace L in pointing and HO actions with a different L
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"] in [1,3]
        and dp["hel_o_belief"] == 0 and ((dp['pt'] and dp['pt_target']==1) or (dp['ho'] and dp['ho_target']==1))):
        #print(dp)
        to_add=[]
        for j in range(3): #2, 5, 1
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(len(to_add)+1)
            dp_aug["utt"] = dp["utt"]
            dp_aug["da"] = dp["da"]
            dp_aug["action"] = dp["action"]
            dp_aug["hel_ot_belief"] = 1
            dp_aug["hel_l_belief"] = 2
            dp_aug["hel_o_belief"] = 0
            dp_aug["pt"] = dp["pt"]
            if dp_aug["pt_target"]==1:
                dp_aug["pt"] = 2 #dp["pt"]
            dp_aug["pt_target"] = dp["pt_target"]
            dp_aug["ho"] = dp["ho"]
            if dp_aug["ho_target"]==1:
                dp_aug["ho"] = 2 #dp["pt"]
            dp_aug["ho_target"] = dp["ho_target"]
            dp_aug["ho_type"] = dp["ho_type"]
            dp_aug["eld_da"] = "Instruct"
            dp_aug["next_actor"] = "eld"
            dp_aug["eld_action"] = 2
            '''e = random.random()
            if e >= 0 and e< 0.6: #(0,0.5), (0,0.7)
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.25 and e< 0.5:
                #dp_aug["eld_action"] = 4 #gives l,ot
            if e>= 0.8 and e <= 1:
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.75 and e<= 1:
                #dp_aug["eld_action"] = 5 #gives l,ot'''
            if not dp_aug in to_add:
                to_add.append(dp_aug)
            #print(dp_aug)
        #print(to_add)
        train_aug.extend(to_add)

    prev_utts = ['i need the ',
                'i am looking for the ',
                'get me the ',
                'bring me the ',
                'i said i wanted the ',
                'i said cabinet number 1',
                'look inside the drawer',
                'check c#2',
                'it should be inside d#4',
                'please check inside d#2']
    #(2,0,0) for prev state
    eps = random.random()
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==0
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']

        train_aug.append(dp_aug)
    #(2,1,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        train_aug.append(dp_aug)
    #(1,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_eld_action']=2
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[9]
        train_aug.append(dp_aug)
    #(2,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=3
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot'] + ". " + prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot'] + ". " + prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot'] + ". " + prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot'] + ". " + prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot'] + ". " + prev_utts[9]
        train_aug.append(dp_aug)
    #(1,1,2) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==1 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=1
        dp_aug['prev_hel_ot_belief']=1
        dp_aug['prev_hel_o_belief']=2
        dp_aug['prev_eld_action']=1
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        dp_aug['pev_da']='Instruct'
        train_aug.append(dp_aug)

In [ ]:
random.seed(345)
valid_aug = []

for i in valid_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    # add (0,0,0) and (0,1,0) instances
    det_ot_utts = ['uh what did you need',
                   'alright what are we looking for',
                   'what are we looking for',
                   'okay what am i looking for',
                   'what do you need to find first']
    #print(trial_idx, move_idx)
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'eld' or dp["prev_da"] not in ["State-y", "State-n"]) and
        (dp["prev_actor"] == 'eld' or "Open" in condensed_corpus[trial_idx]["moves"][move_idx-1][4]) and
        (move_idx < 2 or condensed_corpus[trial_idx]["moves"][move_idx-2][10] != '3')):
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        to_add = []
        det_ot_utts_copy = list(det_ot_utts)
        random.shuffle(det_ot_utts_copy)
        for j in range(2):
            if ((hel_l_belief and random.random() < 0.15) or
                (not hel_l_belief and random.random() < 0.75)):
                if len(det_ot_utts_copy) == 0: continue
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(len(to_add)+1)
                dp_aug["utt"] = det_ot_utts_copy.pop(0)
                dp_aug["da"] = "Query-w"
                dp_aug["action"] = 1
                dp_aug["hel_ot_belief"] = 0
                dp_aug["hel_l_belief"] = hel_l_belief
                dp_aug["hel_o_belief"] = 0
                dp_aug["pt"] = 0
                dp_aug["pt_target"] = 0
                dp_aug["ho"] = 0
                dp_aug["ho_target"] = 0
                dp_aug["ho_type"] = ''
                dp_aug["eld_action"] = 1
                dp_aug["eld_da"] = "Reply-w"
                dp_aug["next_actor"] = "eld"
                to_add.append(dp_aug)
        valid_aug.extend(to_add)


        # add (2,0,0) and (2,1,0) instances
        # take random HEL actions containing the target OT and replace OT in utterance with a random different OT (that isn’t a sub/superclass)
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and dp["prev_actor"] == 'hel'
        and ((re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '2'))):

        #print(dp)
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        x = objs[:]
        #print(dp["ot"])
        #print(x)
        x.remove(dp["ot"])
        to_add = []
        for j in range(50): #50, 100, 150
            if ((hel_l_belief and random.random() < 0.02) or
                (not hel_l_belief and random.random() < 0.9)):
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(0)
                if len(x) >0:
                    ot_aug = x[random.randint(0, len(x)-1)]
                    x.remove(ot_aug)
                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], ot_aug)
                    dp_aug["da"] = dp["da"]
                    dp_aug["action"] = dp_aug["action"]
                    dp_aug["hel_ot_belief"] = 2
                    dp_aug["hel_l_belief"] = hel_l_belief
                    dp_aug["hel_o_belief"] = 0
                    dp_aug["pt"] = dp["pt"]
                    if dp_aug["pt_target"]==2:
                        dp_aug["pt"] = 2 #dp["pt"]
                    dp_aug["pt_target"] = dp["pt_target"]
                    dp_aug["ho"] = dp["ho"]
                    if dp_aug["ho_target"]==2:
                        dp_aug["ho"] = 2 #dp["pt"]
                    dp_aug["ho_target"] = dp["ho_target"]
                    dp_aug["ho_type"] = dp["ho_type"]
                    dp_aug["eld_action"] = 1
                    dp_aug["eld_da"] = "Instruct"
                    dp_aug["next_actor"] = "eld"
                    if not dp_aug in to_add:
                        to_add.append(dp_aug)
                #print(dp_aug)
        #print(to_add)
        valid_aug.extend(to_add)
        # add (1,1,2) instances
    # randomly replace O in pointing and HO actions with a different O
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"]==1
        and (not dp["hel_o_belief"] == 0) and ((dp['pt'] and dp['pt_target']==2) or (dp['ho'] and dp['ho_target']==2))):
        #print(dp)
        to_add=[]
        x = spec_objs[:]
        if dp["o"] in x:
            x.remove(dp["o"])
        to_add=[]
        for j in range(3): #2 #3, 5
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(0)
            if len(x) >0:
                dp_aug["utt"] = dp["utt"]
                o_aug = x[random.randint(0, len(x)-1)]
                x.remove(o_aug)
                if dp["utt"]:
                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], o_aug)
                    dp_aug["utt"] = dp["utt"].replace(dp["o"], o_aug)
                dp_aug["da"] = dp["da"]
                dp_aug["action"] = dp["action"]
                dp_aug["hel_ot_belief"] = 1
                dp_aug["hel_l_belief"] = 1
                dp_aug["hel_o_belief"] = 2
                dp_aug["pt"] = dp["pt"]
                if dp_aug["pt_target"]==2:
                    dp_aug["pt"] = 3 #dp["pt"]
                dp_aug["pt_target"] = dp["pt_target"]
                dp_aug["ho"] = dp["ho"]
                if dp_aug["ho_target"]==2:
                    dp_aug["ho"] = 3 #dp["pt"]
                dp_aug["ho_target"] = dp["ho_target"]
                dp_aug["ho_type"] = dp["ho_type"]
                dp_aug["eld_action"] = 1 #gives spec ot
                dp_aug["eld_da"] = "Instruct"
                dp_aug["next_actor"] = "eld"
                if not dp_aug in to_add:
                    to_add.append(dp_aug)
            #print(dp_aug)
        #print(to_add)
        valid_aug.extend(to_add)    # add (1,2,0) instances
    # randomly replace L in pointing and HO actions with a different L
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"] in [1,3]
        and dp["hel_o_belief"] == 0 and ((dp['pt'] and dp['pt_target']==1) or (dp['ho'] and dp['ho_target']==1))):
        #print(dp)
        to_add=[]
        for j in range(3): #2, 5, 1
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(0)
            dp_aug["utt"] = dp["utt"]
            dp_aug["da"] = dp["da"]
            dp_aug["action"] = dp_aug["action"]
            dp_aug["hel_ot_belief"] = 1
            dp_aug["hel_l_belief"] = 2
            dp_aug["hel_o_belief"] = 0
            dp_aug["pt"] = dp["pt"]
            if dp_aug["pt_target"]==1:
                dp_aug["pt"] = 2 #dp["pt"]
            dp_aug["pt_target"] = dp["pt_target"]
            dp_aug["ho"] = dp["ho"]
            if dp_aug["ho_target"]==1:
                dp_aug["ho"] = 2 #dp["pt"]
            dp_aug["ho_target"] = dp["ho_target"]
            dp_aug["ho_type"] = dp["ho_type"]
            dp_aug["eld_da"] = "Instruct"
            dp_aug["next_actor"] = "eld"
            dp_aug["eld_action"] = 2
            '''e = random.random()
            if e >= 0 and e< 0.6: #(0,0.5), (0,0.7)
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.25 and e< 0.5:
                #dp_aug["eld_action"] = 4 #gives l,ot
            if e>= 0.8 and e <= 1:
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.75 and e<= 1:
                #dp_aug["eld_action"] = 5 #gives l,ot'''
            if not dp_aug in to_add:
                to_add.append(dp_aug)
            #print(dp_aug)
        #print(to_add)
        valid_aug.extend(to_add)

    prev_utts = ['i need the ',
                'i am looking for the ',
                'get me the ',
                'bring me the ',
                'i said i wanted the ',
                'i said cabinet number 1',
                'look inside the drawer',
                'check c#2',
                'it should be inside d#4',
                'please check inside d#2']
    #(2,0,0) for prev state
    eps = random.random()
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==0
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']

        valid_aug.append(dp_aug)
    #(2,1,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        valid_aug.append(dp_aug)
    #(1,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_eld_action']=2
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[9]
        valid_aug.append(dp_aug)
    #(2,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=3
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot'] + ". " + prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot'] + ". " + prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot'] + ". " + prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot'] + ". " + prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot'] + ". " + prev_utts[9]
        valid_aug.append(dp_aug)
    #(1,1,2) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==1 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=1
        dp_aug['prev_hel_ot_belief']=1
        dp_aug['prev_hel_o_belief']=2
        dp_aug['prev_eld_action']=1
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        dp_aug['pev_da']='Instruct'
        valid_aug.append(dp_aug)

In [ ]:
random.seed(345)
test_aug = []

for i in test_idxs:
    dp = data[i]
    for j,trial in enumerate(condensed_corpus):
        trial_id, move_id = dp["id"].split('|')
        if trial["trial_id"] == trial_id:
            trial_idx = j
            move_idx = int(move_id)-1
            break
    else:
        print("couldn't find datapoint", dp)

    # add (0,0,0) and (0,1,0) instances
    det_ot_utts = ['uh what did you need',
                   'alright what are we looking for',
                   'what are we looking for',
                   'okay what am i looking for',
                   'what do you need to find first']
    #print(trial_idx, move_idx)
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and
        (dp["prev_actor"] == 'eld' or dp["prev_da"] not in ["State-y", "State-n"]) and
        (dp["prev_actor"] == 'eld' or "Open" in condensed_corpus[trial_idx]["moves"][move_idx-1][4]) and
        (move_idx < 2 or condensed_corpus[trial_idx]["moves"][move_idx-2][10] != '3')):
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        to_add = []
        det_ot_utts_copy = list(det_ot_utts)
        random.shuffle(det_ot_utts_copy)
        for j in range(2):
            if ((hel_l_belief and random.random() < 0.15) or
                (not hel_l_belief and random.random() < 0.75)):
                if len(det_ot_utts_copy) == 0: continue
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(0) #".{}".format(len(to_add)+1)
                dp_aug["utt"] = det_ot_utts_copy.pop(0)
                dp_aug["da"] = "Query-w"
                dp_aug["action"] = 1
                dp_aug["hel_ot_belief"] = 0
                dp_aug["hel_l_belief"] = hel_l_belief
                dp_aug["hel_o_belief"] = 0
                dp_aug["pt"] = 0
                dp_aug["pt_target"] = 0
                dp_aug["ho"] = 0
                dp_aug["ho_target"] = 0
                dp_aug["ho_type"] = ''
                dp_aug["eld_action"] = 1
                dp_aug["eld_da"] = "Reply-w"
                dp_aug["next_actor"] = "eld"
                to_add.append(dp_aug)
        test_aug.extend(to_add)


        # add (2,0,0) and (2,1,0) instances
        # take random HEL actions containing the target OT and replace OT in utterance with a random different OT (that isn’t a sub/superclass)
    if (move_idx > 0 and
        dp["prev_hel_o_belief"] not in [1, 3] and dp["prev_actor"] == 'hel'
        and ((re.search(dp['ot'], dp['utt']) or condensed_corpus[trial_idx]["moves"][move_idx-2][10] == '2'))):

        #print(dp)
        hel_l_belief = dp["prev_l_given"] or dp["prev_hel_l_belief"]
        x = objs[:]
        #print(dp["ot"])
        #print(x)
        x.remove(dp["ot"])

        to_add = []
        for j in range(50): #50, 100, 150
            if ((hel_l_belief and random.random() < 0.02) or
                (not hel_l_belief and random.random() < 0.9)):
                dp_aug = dict(dp)
                dp_aug["id"] += ".{}".format(0)#".{}".format(len(to_add)+1)
                if len(x) >0:
                    ot_aug = x[random.randint(0, len(x)-1)]
                    x.remove(ot_aug)
                    #print(ot_aug)
                    #print(x)

                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], ot_aug)
                    dp_aug["da"] = dp["da"]
                    dp_aug["action"] = dp_aug["action"]
                    dp_aug["hel_ot_belief"] = 2
                    dp_aug["hel_l_belief"] = hel_l_belief
                    dp_aug["hel_o_belief"] = 0
                    dp_aug["pt"] = dp["pt"]
                    if dp_aug["pt_target"]==2:
                        dp_aug["pt"] = 2 #dp["pt"]
                    dp_aug["pt_target"] = dp["pt_target"]
                    dp_aug["ho"] = dp["ho"]
                    if dp_aug["ho_target"]==2:
                        dp_aug["ho"] = 2 #dp["pt"]
                    dp_aug["ho_target"] = dp["ho_target"]
                    dp_aug["ho_type"] = dp["ho_type"]
                    dp_aug["eld_action"] = 1
                    dp_aug["eld_da"] = "Instruct"
                    dp_aug["next_actor"] = "eld"
                    if not dp_aug in to_add:
                        to_add.append(dp_aug)
                #print(dp_aug)
        #print(to_add)
        test_aug.extend(to_add)
    # add (1,1,2) instances
    # randomly replace O in pointing and HO actions with a different O
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"]==1
        and (not dp["hel_o_belief"] == 0) and ((dp['pt'] and dp['pt_target']==2) or (dp['ho'] and dp['ho_target']==2))):
        #print(dp)
        to_add=[]
        x = spec_objs[:]
        if dp["o"] in x:
            x.remove(dp["o"])
        for j in range(3): #2 #3, 5
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(0)
            if len(x) >0:
                dp_aug["utt"] = dp["utt"]
                o_aug = x[random.randint(0, len(x)-1)]
                x.remove(o_aug)
                if dp["utt"]:
                    dp_aug["utt"] = dp["utt"].replace(dp["ot"], o_aug)
                    dp_aug["utt"] = dp["utt"].replace(dp["o"], o_aug)
                dp_aug["da"] = dp["da"]
                dp_aug["action"] = dp["action"]
                dp_aug["hel_ot_belief"] = 1
                dp_aug["hel_l_belief"] = 1
                dp_aug["hel_o_belief"] = 2
                dp_aug["pt"] = dp["pt"]
                if dp_aug["pt_target"]==2:
                    dp_aug["pt"] = 3 #dp["pt"]
                dp_aug["pt_target"] = dp["pt_target"]
                dp_aug["ho"] = dp["ho"]
                if dp_aug["ho_target"]==2:
                    dp_aug["ho"] = 3 #dp["pt"]
                dp_aug["ho_target"] = dp["ho_target"]
                dp_aug["ho_type"] = dp["ho_type"]
                dp_aug["eld_action"] = 1 #gives spec ot
                dp_aug["eld_da"] = "Instruct"
                dp_aug["next_actor"] = "eld"
                if not dp_aug in to_add:
                    to_add.append(dp_aug)
            #print(dp_aug)
        #print(to_add)
        test_aug.extend(to_add)

    # add (1,2,0) instances
    # randomly replace L in pointing and HO actions with a different L
    if (move_idx > 0 and dp["prev_actor"] == 'hel' and dp["hel_ot_belief"]==1 and dp["hel_l_belief"] in [1,3]
        and dp["hel_o_belief"] == 0 and ((dp['pt'] and dp['pt_target']==1) or (dp['ho'] and dp['ho_target']==1))):
        #print(dp)
        to_add=[]
        for j in range(3): #2, 5, 1
            dp_aug = dict(dp)
            dp_aug["id"] += ".{}".format(0) #dp_aug["id"] += ".{}".format(len(to_add)+1)
            dp_aug["utt"] = dp["utt"]
            dp_aug["da"] = dp["da"]
            dp_aug["action"] = dp_aug["action"]
            dp_aug["hel_ot_belief"] = 1
            dp_aug["hel_l_belief"] = 2
            dp_aug["hel_o_belief"] = 0
            dp_aug["pt"] = dp["pt"]
            if dp_aug["pt_target"]==1:
                dp_aug["pt"] = 2 #dp["pt"]
            dp_aug["pt_target"] = dp["pt_target"]
            dp_aug["ho"] = dp["ho"]
            if dp_aug["ho_target"]==1:
                dp_aug["ho"] = 2 #dp["pt"]
            dp_aug["ho_target"] = dp["ho_target"]
            dp_aug["ho_type"] = dp["ho_type"]
            dp_aug["eld_da"] = "Instruct"
            dp_aug["next_actor"] = "eld"
            dp_aug["eld_action"] = 2
            '''e = random.random()
            if e >= 0 and e< 0.6: #(0,0.5), (0,0.7)
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.25 and e< 0.5:
                #dp_aug["eld_action"] = 4 #gives l,ot
            if e>= 0.8 and e <= 1:
                dp_aug["eld_action"] = 2 #gives l
            #elif e>= 0.75 and e<= 1:
                #dp_aug["eld_action"] = 5 #gives l,ot'''
            if not dp_aug in to_add:
                to_add.append(dp_aug)
            #print(dp_aug)
        #print(to_add)
        test_aug.extend(to_add)

    prev_utts = ['i need the ',
                'i am looking for the ',
                'get me the ',
                'bring me the ',
                'i said i wanted the ',
                'i said cabinet number 1',
                'look inside the drawer',
                'check c#2',
                'it should be inside d#4',
                'please check inside d#2']
    #(2,0,0) for prev state
    eps = random.random()
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==0
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']

        test_aug.append(dp_aug)
    #(2,1,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=1
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        test_aug.append(dp_aug)
    #(1,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_eld_action']=2
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[9]
        test_aug.append(dp_aug)
    #(2,2,0) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==0 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=2
        dp_aug['prev_hel_ot_belief']=2
        dp_aug['prev_eld_action']=3
        dp_aug['pev_da']='Instruct'
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot'] + ". " + prev_utts[5]
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot'] + ". " + prev_utts[6]
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot'] + ". " + prev_utts[7]
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot'] + ". " + prev_utts[8]
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot'] + ". " + prev_utts[9]
        test_aug.append(dp_aug)
    #(1,1,2) for prev state
    if (move_idx > 0 and dp['prev_hel_ot_belief']==1 and dp['prev_hel_l_belief']==1
        and dp['prev_hel_o_belief']==1 and dp['prev_actor']=='eld'):
        dp_aug = dp.copy()
        dp_aug['prev_hel_l_belief']=1
        dp_aug['prev_hel_ot_belief']=1
        dp_aug['prev_hel_o_belief']=2
        dp_aug['prev_eld_action']=1
        if eps<0.2:
            dp_aug['prev_utt'] = prev_utts[0] + dp['ot']
        elif eps>=0.2 and eps<0.4:
            dp_aug['prev_utt'] = prev_utts[1] + dp['ot']
        elif eps>=0.4 and eps<0.6:
            dp_aug['prev_utt'] = prev_utts[2] + dp['ot']
        elif eps>=0.6 and eps<0.8:
            dp_aug['prev_utt'] = prev_utts[3] + dp['ot']
        elif eps>0.8:
            dp_aug['prev_utt'] = prev_utts[4] + dp['ot']
        dp_aug['pev_da']='Instruct'
        test_aug.append(dp_aug)

In [ ]:
states = {(i,j,k):0 for i in range(3) for j in range(4) for k in range(4)}
for d in [data[i] for i in train_idxs]+train_aug:
    states[(d["prev_hel_ot_belief"], d["prev_hel_l_belief"], d["prev_hel_o_belief"])] += 1
states

{(0, 0, 0): 158,
 (0, 0, 1): 0,
 (0, 0, 2): 0,
 (0, 0, 3): 0,
 (0, 1, 0): 4,
 (0, 1, 1): 0,
 (0, 1, 2): 0,
 (0, 1, 3): 0,
 (0, 2, 0): 0,
 (0, 2, 1): 0,
 (0, 2, 2): 0,
 (0, 2, 3): 0,
 (0, 3, 0): 0,
 (0, 3, 1): 0,
 (0, 3, 2): 0,
 (0, 3, 3): 0,
 (1, 0, 0): 271,
 (1, 0, 1): 2,
 (1, 0, 2): 4,
 (1, 0, 3): 0,
 (1, 1, 0): 437,
 (1, 1, 1): 38,
 (1, 1, 2): 25,
 (1, 1, 3): 0,
 (1, 2, 0): 89,
 (1, 2, 1): 0,
 (1, 2, 2): 0,
 (1, 2, 3): 0,
 (1, 3, 0): 0,
 (1, 3, 1): 0,
 (1, 3, 2): 0,
 (1, 3, 3): 0,
 (2, 0, 0): 126,
 (2, 0, 1): 0,
 (2, 0, 2): 0,
 (2, 0, 3): 0,
 (2, 1, 0): 89,
 (2, 1, 1): 0,
 (2, 1, 2): 0,
 (2, 1, 3): 0,
 (2, 2, 0): 89,
 (2, 2, 1): 0,
 (2, 2, 2): 0,
 (2, 2, 3): 0,
 (2, 3, 0): 0,
 (2, 3, 1): 0,
 (2, 3, 2): 0,
 (2, 3, 3): 0}

In [ ]:
# Extract data points with the correct fields including prev_eld_da as specified
training_data= []
for trial in [data[i] for i in train_idxs]+train_aug:

        if trial["actor"] != "hel":
            continue

        # # Find previous helper move and elder utterance
        # prev_hel_move = None
        # prev_eld_utt = ""
        # prev_eld_da = ""
        # for j in range(i-1, -1, -1):
        #     if trial["moves"][j][0] == 'hel' and prev_hel_move is None:
        #         prev_hel_move = trial["moves"][j]
        #     elif trial["moves"][j][0] == 'eld':
        #         prev_eld_utt = trial["moves"][j][1]
        #         prev_eld_da = trial["moves"][j][2]
        #         break

        # if prev_hel_move is None:
        #     continue
        if trial["actor"] == "hel":
            datapoint = {
              "prev_utt": trial["prev_utt"],
              "current_utt": trial["utt"],
              "elder_utt": trial["prev_utt"],
              "prev_hel_belief": {
                  "ot_belief": trial['prev_hel_ot_belief'],
                  "l_belief": trial['prev_hel_l_belief'],
                  "o_belief": trial['prev_hel_o_belief']
              },
              "hel_belief": {
                  "ot_belief": trial['hel_ot_belief'],
                  "l_belief": trial['hel_l_belief'],
                  "o_belief": trial['hel_o_belief']
              },

              "da": trial["da"],
              "hel_action": trial["action"],
              "prev_eld_da": trial["prev_da"],
              "prev_eld_action": trial["eld_action"]
          }

        elif trial["actor"] == "eld":
            datapoint = {
              "prev_utt": trial["prev_utt"],
              "current_utt": trial["utt"],
              "elder_utt": trial["prev_utt"],
              "prev_hel_belief": {
                  "ot_belief": trial['prev_hel_ot_belief'],
                  "l_belief": trial['prev_hel_l_belief'],
                  "o_belief": trial['prev_hel_o_belief']
              },
              "hel_belief": {
                  "ot_belief": trial['hel_ot_belief'],
                  "l_belief": trial['hel_l_belief'],
                  "o_belief": trial['hel_o_belief']
              },

              "da": trial["da"],
              "hel_action": trial["action"],
              "prev_eld_da": trial["prev_da"],
              "prev_eld_action": trial["eld_action"]
          }

        training_data.append(datapoint)

# Save to JSON
json_file_path = 'training_data.json'
with open(json_file_path, 'w') as json_file:
    json.dump(training_data, json_file, indent=4)

IndentationError: expected an indented block after 'elif' statement on line 43 (<ipython-input-28-42349bb6868c>, line 44)

In [ ]:
train_aug[2]

{'id': '2010-11-04-002-75.txt|9.2',
 'ot': 'glass',
 'utt': 'okay what am i looking for',
 'da': 'Query-w',
 'action': 1,
 'actor': 'hel',
 'next_actor': 'eld',
 'prev_utt': "no it's just some cups i don't see any glasses",
 'prev_da': 'State-n',
 'prev_actor': 'eld',
 'prev_ot_given': 1,
 'prev_l_given': 0,
 'prev_hel_ot_belief': 1,
 'prev_hel_l_belief': 0,
 'prev_hel_o_belief': 0,
 'o': 'cup#1',
 'pt_target': 0,
 'pt': 0,
 'ho_target': 0,
 'ho': 0,
 'ho_type': '',
 'prev_eld_action': 6,
 'hel_ot_belief': 0,
 'hel_l_belief': 0,
 'hel_o_belief': 0,
 'eld_action': 1,
 'eld_da': 'Reply-w',
 'eld_sees_o': 2}

In [ ]:
len([data[i] for i in train_idxs]+train_aug)

1332

In [ ]:
len([data[i] for i in valid_idxs]+valid_aug)

281

In [ ]:
len([data[i] for i in test_idxs]+test_aug)

299

292

In [ ]:
for i in valid_idxs:
  print(data[i])

{'id': '2010-10-21-002-51.txt|9', 'ot': 'salad bowl', 'utt': 'action', 'da': '', 'action': 0, 'actor': 'hel', 'next_actor': 'hel', 'prev_utt': 'the left', 'prev_da': 'Reply-w', 'prev_actor': 'eld', 'prev_ot_given': 1, 'prev_l_given': 1, 'prev_hel_ot_belief': 1, 'prev_hel_l_belief': 0, 'prev_hel_o_belief': 0, 'o': 'bowl#1', 'pt_target': 0, 'pt': 0, 'ho_target': 1, 'ho': 1, 'ho_type': 'Open', 'prev_eld_action': 2, 'hel_ot_belief': 1, 'hel_l_belief': 1, 'hel_o_belief': 0, 'eld_action': 0, 'eld_da': '', 'eld_sees_o': 0}
{'id': '2010-10-21-002-49.txt|9', 'ot': 'cup', 'utt': "i don't see a cup in here", 'da': 'State-n', 'action': 8, 'actor': 'hel', 'next_actor': 'eld', 'prev_utt': 'action', 'prev_da': '', 'prev_actor': 'hel', 'prev_ot_given': 1, 'prev_l_given': 1, 'prev_hel_ot_belief': 1, 'prev_hel_l_belief': 1, 'prev_hel_o_belief': 0, 'o': 'cup#2', 'pt_target': 0, 'pt': 0, 'ho_target': 1, 'ho': 1, 'ho_type': 'Close', 'prev_eld_action': 0, 'hel_ot_belief': 1, 'hel_l_belief': 0, 'hel_o_belief

In [ ]:
train_set = []

for trial in [data[i] for i in train_idxs]+train_aug:
   train_set.append(trial)

In [ ]:
len(train_set)

1332

In [ ]:
valid_set = []

for trial in [data[i] for i in valid_idxs]+valid_aug:
   valid_set.append(trial)

In [ ]:
len(valid_set)

281

In [ ]:
test_set = []

for trial in [data[i] for i in test_idxs]+test_aug:
   test_set.append(trial)

In [ ]:
len(test_set)

299

In [ ]:
lines = list(train_set)
len(lines)

1332

In [ ]:
lines

[{'id': '2010-11-11-001-ep02-92.txt|5',
  'ot': 'grapes',
  'utt': 'there are some paper towels where else is the',
  'da': 'Query-w',
  'action': 2,
  'actor': 'hel',
  'next_actor': 'eld',
  'prev_utt': 'in here uh-huh',
  'prev_da': 'Acknowledge',
  'prev_actor': 'hel',
  'prev_ot_given': 1,
  'prev_l_given': 1,
  'prev_hel_ot_belief': 1,
  'prev_hel_l_belief': 1,
  'prev_hel_o_belief': 0,
  'o': 'grapes',
  'pt_target': 0,
  'pt': 0,
  'ho_target': 1,
  'ho': 2,
  'ho_type': 'Close',
  'prev_eld_action': 0,
  'hel_ot_belief': 1,
  'hel_l_belief': 0,
  'hel_o_belief': 0,
  'eld_action': 2,
  'eld_da': 'Reply-w',
  'eld_sees_o': 0},
 {'id': '2010-11-04-002-75.txt|9',
  'ot': 'glass',
  'utt': 'ok',
  'da': 'Acknowledge',
  'action': 6,
  'actor': 'hel',
  'next_actor': 'eld',
  'prev_utt': "no it's just some cups i don't see any glasses",
  'prev_da': 'State-n',
  'prev_actor': 'eld',
  'prev_ot_given': 1,
  'prev_l_given': 0,
  'prev_hel_ot_belief': 1,
  'prev_hel_l_belief': 0,
  'p

In [ ]:
lines[1]

{'id': '2010-11-04-002-75.txt|9',
 'ot': 'glass',
 'utt': 'ok',
 'da': 'Acknowledge',
 'action': 6,
 'actor': 'hel',
 'next_actor': 'eld',
 'prev_utt': "no it's just some cups i don't see any glasses",
 'prev_da': 'State-n',
 'prev_actor': 'eld',
 'prev_ot_given': 1,
 'prev_l_given': 0,
 'prev_hel_ot_belief': 1,
 'prev_hel_l_belief': 0,
 'prev_hel_o_belief': 0,
 'o': 'cup#1',
 'pt_target': 0,
 'pt': 0,
 'ho_target': 0,
 'ho': 0,
 'ho_type': '',
 'prev_eld_action': 6,
 'hel_ot_belief': 1,
 'hel_l_belief': 0,
 'hel_o_belief': 0,
 'eld_action': 0,
 'eld_da': 'Instruct',
 'eld_sees_o': 2}

In [ ]:
train_corpus = []  # Structure: [{trial_id: str, ot: str, o: str, init_action: int, moves: [...]}]

i = 0
while i < len(lines):
    line = lines[i]

    # Extract trial-level details
    trial_id = line['id']
    ot = line['ot'].lower() if 'ot' in line and line['ot'] else None
    o = line['o'].lower() if 'o' in line and line['o'] else None
    init_action = line['action'] if 'action' in line else 0
    init_sees = line['eld_sees_o'] if 'eld_sees_o' in line else 0

    if not o:  # Skip to next trial if `o` is missing
        i += 1
        while i < len(lines) and 'id' in lines[i]:  # Look for the next trial
            i += 1
        continue

    moves = []
    while i < len(lines) and 'id' in lines[i]:  # Parse moves within the trial
        move = lines[i]

        # Handle missing `eld_action`
        if move['actor'] == 'hel' and move.get('eld_action', '') == '':
            move['eld_action'] = 0

        # Handle missing `action` for helper
        if move['actor'] == 'hel' and move.get('action', '') == '':
            if i > 0 and lines[i - 1]['actor'] == 'hel' and 'action' in lines[i - 1]:
                move['action'] = lines[i - 1]['action']
            elif i + 1 < len(lines) and lines[i + 1]['actor'] == 'hel' and 'action' in lines[i + 1]:
                move['action'] = lines[i + 1]['action']
            else:
                move['action'] = 6  # Default to 6

        # Handle missing `eld_sees_o`
        if 'eld_sees_o' not in move or move['eld_sees_o'] == '':
            if moves:
                move['eld_sees_o'] = moves[-1]['eld_sees_o']
            else:
                move['eld_sees_o'] = init_sees

        moves.append({
            "actor": move['actor'],
            "utterance": move.get('utt', ''),
            "da": move.get('da', ''),
            "pt": move.get('pt', 0),
            "ho": move.get('ho', 0),
            "eldL": move.get('pt_target', 0),
            "eldOtGiven": move.get('prev_ot_given', 0),
            "eldLGiven": move.get('prev_l_given', 0),
            "helOtBelief": move.get('prev_hel_ot_belief', 0),
            "helLBelief": move.get('prev_hel_l_belief', 0),
            "helOBelief": move.get('prev_hel_o_belief', 0),
            "eldAction": move.get('eld_action', 0),
            "eldSeesO": move.get('eld_sees_o', 0)
        })

        i += 1

    train_corpus.append({
        "trial_id": trial_id,
        "ot": ot,
        "o": o,
        "init_action": init_action,
        "moves": moves
    })

    i += 1  # Move to the next trial


In [ ]:
train_corpus[0]

{'trial_id': '2010-11-11-001-ep02-92.txt|5',
 'ot': 'grapes',
 'o': 'grapes',
 'init_action': 2,
 'moves': [{'actor': 'hel',
   'utterance': 'there are some paper towels where else is the',
   'da': 'Query-w',
   'pt': 0,
   'ho': 2,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 1,
   'helLBelief': 1,
   'helOBelief': 0,
   'eldAction': 2,
   'eldSeesO': 0},
  {'actor': 'hel',
   'utterance': 'ok',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 0,
   'helOtBelief': 1,
   'helLBelief': 0,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 2},
  {'actor': 'hel',
   'utterance': 'okay',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 0,
   'helLBelief': 0,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 1},
  {'actor': 'hel',
   'utterance': 'ok',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eld

In [ ]:
# train_condensed_corpus = []  # combine consecutive moves by same actor if state doesn't change
# for trial in train_corpus:
#     moves = (
#         []
#     )  # moves: [(actor, utterance, da, pt, ho, eldL, eldOtGiven, eldLGiven, helOtBelief, helLBelief, helOBelief, eldAction, eldSeesO)]
#     for move in trial["moves"]:
#         if len(moves) == 0:
#             moves.append(move)
#         else:
#             prev_move = moves[-1]
#             if prev_move[0] == move[0] and move[6:11] == prev_move[6:11]:
#                 # if move and prev_move both have pt/HO, consider as separate moves
#                 if (move[3] or move[4]) and (prev_move[3] or prev_move[4]):
#                     moves.append(move)
#                     continue
#                 prev_move = moves.pop(-1)
#                 new_move = []
#                 # actors should be the same
#                 new_move.append(move[0])
#                 # concatenate utterances
#                 new_move.append(" ".join([prev_move[1], move[1]]).strip())
#                 # take latest DA unless it's 'Explain'
#                 new_move.append(
#                     prev_move[2]
#                     if prev_move[2] and (not move[2] or move[2] == "Explain")
#                     else move[2]
#                 )
#                 # take latest existing pt/ho
#                 new_move.append(
#                     prev_move[3] if prev_move[3] and not move[3] else move[3]
#                 )
#                 new_move.append(
#                     prev_move[4] if prev_move[4] and not move[4] else move[4]
#                 )
#                 # take latest L, eldAction, eldSeesO
#                 # states should be the same
#                 new_move.extend(move[5:])
#                 new_move = tuple(new_move)
#                 moves.append(new_move)
#             else:
#                 moves.append(move)
#     train_condensed_corpus.append(
#         {
#             "trial_id": trial["trial_id"],
#             "ot": trial["ot"],
#             "o": trial["o"],
#             "init_action": trial["init_action"],
#             "moves": moves,
#         }
#     )

In [ ]:
# train_condensed_corpus = []  # Combine consecutive moves by the same actor if state doesn't change

# for trial in train_corpus:
#     condensed_moves = []  # Initialize condensed moves list

#     for move in trial["moves"]:
#         if not condensed_moves:  # First move in trial
#             condensed_moves.append(move)
#         else:
#             prev_move = condensed_moves[-1]

#             # Check if actor and state are unchanged
#             same_actor = prev_move[0] == move[0]
#             same_state = all(
#                 prev_move[i] == move[i] for i in range(6, 11)
#             )  # Compare specific state indices

#             # Treat moves with pt/ho as separate
#             if (move[3] or move[4]) and (prev_move[3] or prev_move[4]):
#                 condensed_moves.append(move)
#                 continue

#             if same_actor and same_state:  # Condense moves
#                 # Combine utterances
#                 concatenated_utterance = " ".join([prev_move[1], move[1]]).strip()
#                 latest_da = (
#                     prev_move[2]
#                     if prev_move[2] and (not move[2] or move[2] == "Explain")
#                     else move[2]
#                 )
#                 latest_pt = prev_move[3] if prev_move[3] and not move[3] else move[3]
#                 latest_ho = prev_move[4] if prev_move[4] and not move[4] else move[4]

#                 # Update previous move
#                 condensed_moves[-1] = (
#                     move[0],  # Actor
#                     concatenated_utterance,  # Combined utterance
#                     latest_da,  # Latest DA
#                     latest_pt,  # Latest pt
#                     latest_ho,  # Latest ho
#                     *move[5:]  # Unchanged state
#                 )
#             else:
#                 condensed_moves.append(move)

#     # Add condensed trial to the corpus
#     train_condensed_corpus.append(
#         {
#             "trial_id": trial["trial_id"],
#             "ot": trial["ot"],
#             "o": trial["o"],
#             "init_action": trial["init_action"],
#             "moves": condensed_moves,
#         }
#     )


In [ ]:
train_condensed_corpus = []  # Combine consecutive moves by the same actor if state doesn't change

for trial in train_corpus:
    condensed_moves = []  # Initialize condensed moves list

    for move in trial["moves"]:
        if not condensed_moves:  # First move in trial
            condensed_moves.append(move)
        else:
            prev_move = condensed_moves[-1]

            # Check if actor and state are unchanged
            same_actor = prev_move["actor"] == move["actor"]
            same_state = (
                prev_move["pt"] == move["pt"]
                and prev_move["ho"] == move["ho"]
                and prev_move["eldL"] == move["eldL"]
                and prev_move["eldOtGiven"] == move["eldOtGiven"]
                and prev_move["eldLGiven"] == move["eldLGiven"]
                and prev_move["helOtBelief"] == move["helOtBelief"]
                and prev_move["helLBelief"] == move["helLBelief"]
                and prev_move["helOBelief"] == move["helOBelief"]
                and prev_move["eldAction"] == move["eldAction"]
                and prev_move["eldSeesO"] == move["eldSeesO"]
            )

            # Treat moves with pt/ho as separate
            if (move["pt"] or move["ho"]) and (prev_move["pt"] or prev_move["ho"]):
                condensed_moves.append(move)
                continue

            if same_actor and same_state:  # Condense moves
                # Combine utterances
                concatenated_utterance = " ".join([prev_move["utterance"], move["utterance"]]).strip()
                latest_da = (
                    prev_move["da"]
                    if prev_move["da"] and (not move["da"] or move["da"] == "Explain")
                    else move["da"]
                )
                latest_pt = prev_move["pt"] if prev_move["pt"] and not move["pt"] else move["pt"]
                latest_ho = prev_move["ho"] if prev_move["ho"] and not move["ho"] else move["ho"]

                # Update previous move
                condensed_moves[-1] = {
                    "actor": move["actor"],
                    "utterance": concatenated_utterance,
                    "da": latest_da,
                    "pt": latest_pt,
                    "ho": latest_ho,
                    "eldL": move["eldL"],
                    "eldOtGiven": move["eldOtGiven"],
                    "eldLGiven": move["eldLGiven"],
                    "helOtBelief": move["helOtBelief"],
                    "helLBelief": move["helLBelief"],
                    "helOBelief": move["helOBelief"],
                    "eldAction": move["eldAction"],
                    "eldSeesO": move["eldSeesO"],
                }
            else:
                condensed_moves.append(move)

    # Add condensed trial to the corpus
    train_condensed_corpus.append(
        {
            "trial_id": trial["trial_id"],
            "ot": trial["ot"],
            "o": trial["o"],
            "init_action": trial["init_action"],
            "moves": condensed_moves,
        }
    )


In [ ]:
train_condensed_corpus[0]

{'trial_id': '2010-11-11-001-ep02-92.txt|5',
 'ot': 'grapes',
 'o': 'grapes',
 'init_action': 2,
 'moves': [{'actor': 'hel',
   'utterance': 'there are some paper towels where else is the',
   'da': 'Query-w',
   'pt': 0,
   'ho': 2,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 1,
   'helLBelief': 1,
   'helOBelief': 0,
   'eldAction': 2,
   'eldSeesO': 0},
  {'actor': 'hel',
   'utterance': 'ok',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 0,
   'helOtBelief': 1,
   'helLBelief': 0,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 2},
  {'actor': 'hel',
   'utterance': 'okay',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 0,
   'helLBelief': 0,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 1},
  {'actor': 'hel',
   'utterance': 'ok',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eld

In [ ]:
lines = list(valid_set)
len(lines)

281

In [ ]:
valid_corpus = []  # Structure: [{trial_id: str, ot: str, o: str, init_action: int, moves: [...]}]

i = 0
while i < len(lines):
    line = lines[i]

    # Extract trial-level details
    trial_id = line['id']
    ot = line['ot'].lower() if 'ot' in line and line['ot'] else None
    o = line['o'].lower() if 'o' in line and line['o'] else None
    init_action = line['action'] if 'action' in line else 0
    init_sees = line['eld_sees_o'] if 'eld_sees_o' in line else 0

    if not o:  # Skip to next trial if `o` is missing
        i += 1
        while i < len(lines) and 'id' in lines[i]:  # Look for the next trial
            i += 1
        continue

    moves = []
    while i < len(lines) and 'id' in lines[i]:  # Parse moves within the trial
        move = lines[i]

        # Handle missing `eld_action`
        if move['actor'] == 'hel' and move.get('eld_action', '') == '':
            move['eld_action'] = 0

        # Handle missing `action` for helper
        if move['actor'] == 'hel' and move.get('action', '') == '':
            if i > 0 and lines[i - 1]['actor'] == 'hel' and 'action' in lines[i - 1]:
                move['action'] = lines[i - 1]['action']
            elif i + 1 < len(lines) and lines[i + 1]['actor'] == 'hel' and 'action' in lines[i + 1]:
                move['action'] = lines[i + 1]['action']
            else:
                move['action'] = 6  # Default to 6

        # Handle missing `eld_sees_o`
        if 'eld_sees_o' not in move or move['eld_sees_o'] == '':
            if moves:
                move['eld_sees_o'] = moves[-1]['eld_sees_o']
            else:
                move['eld_sees_o'] = init_sees

        moves.append({
            "actor": move['actor'],
            "utterance": move.get('utt', ''),
            "da": move.get('da', ''),
            "pt": move.get('pt', 0),
            "ho": move.get('ho', 0),
            "eldL": move.get('pt_target', 0),
            "eldOtGiven": move.get('prev_ot_given', 0),
            "eldLGiven": move.get('prev_l_given', 0),
            "helOtBelief": move.get('prev_hel_ot_belief', 0),
            "helLBelief": move.get('prev_hel_l_belief', 0),
            "helOBelief": move.get('prev_hel_o_belief', 0),
            "eldAction": move.get('eld_action', 0),
            "eldSeesO": move.get('eld_sees_o', 0)
        })

        i += 1

    valid_corpus.append({
        "trial_id": trial_id,
        "ot": ot,
        "o": o,
        "init_action": init_action,
        "moves": moves
    })

    i += 1  # Move to the next trial


In [ ]:
valid_condensed_corpus = []  # Combine consecutive moves by the same actor if state doesn't change

for trial in valid_corpus:
    condensed_moves = []  # Initialize condensed moves list

    for move in trial["moves"]:
        if not condensed_moves:  # First move in trial
            condensed_moves.append(move)
        else:
            prev_move = condensed_moves[-1]

            # Check if actor and state are unchanged
            same_actor = prev_move["actor"] == move["actor"]
            same_state = (
                prev_move["pt"] == move["pt"]
                and prev_move["ho"] == move["ho"]
                and prev_move["eldL"] == move["eldL"]
                and prev_move["eldOtGiven"] == move["eldOtGiven"]
                and prev_move["eldLGiven"] == move["eldLGiven"]
                and prev_move["helOtBelief"] == move["helOtBelief"]
                and prev_move["helLBelief"] == move["helLBelief"]
                and prev_move["helOBelief"] == move["helOBelief"]
                and prev_move["eldAction"] == move["eldAction"]
                and prev_move["eldSeesO"] == move["eldSeesO"]
            )

            # Treat moves with pt/ho as separate
            if (move["pt"] or move["ho"]) and (prev_move["pt"] or prev_move["ho"]):
                condensed_moves.append(move)
                continue

            if same_actor and same_state:  # Condense moves
                # Combine utterances
                concatenated_utterance = " ".join([prev_move["utterance"], move["utterance"]]).strip()
                latest_da = (
                    prev_move["da"]
                    if prev_move["da"] and (not move["da"] or move["da"] == "Explain")
                    else move["da"]
                )
                latest_pt = prev_move["pt"] if prev_move["pt"] and not move["pt"] else move["pt"]
                latest_ho = prev_move["ho"] if prev_move["ho"] and not move["ho"] else move["ho"]

                # Update previous move
                condensed_moves[-1] = {
                    "actor": move["actor"],
                    "utterance": concatenated_utterance,
                    "da": latest_da,
                    "pt": latest_pt,
                    "ho": latest_ho,
                    "eldL": move["eldL"],
                    "eldOtGiven": move["eldOtGiven"],
                    "eldLGiven": move["eldLGiven"],
                    "helOtBelief": move["helOtBelief"],
                    "helLBelief": move["helLBelief"],
                    "helOBelief": move["helOBelief"],
                    "eldAction": move["eldAction"],
                    "eldSeesO": move["eldSeesO"],
                }
            else:
                condensed_moves.append(move)

    # Add condensed trial to the corpus
    valid_condensed_corpus.append(
        {
            "trial_id": trial["trial_id"],
            "ot": trial["ot"],
            "o": trial["o"],
            "init_action": trial["init_action"],
            "moves": condensed_moves,
        }
    )


In [ ]:
lines = list(test_set)
len(lines)

299

In [ ]:
test_corpus = []  # Structure: [{trial_id: str, ot: str, o: str, init_action: int, moves: [...]}]

i = 0
while i < len(lines):
    line = lines[i]

    # Extract trial-level details
    trial_id = line['id']
    ot = line['ot'].lower() if 'ot' in line and line['ot'] else None
    o = line['o'].lower() if 'o' in line and line['o'] else None
    init_action = line['action'] if 'action' in line else 0
    init_sees = line['eld_sees_o'] if 'eld_sees_o' in line else 0

    if not o:  # Skip to next trial if `o` is missing
        i += 1
        while i < len(lines) and 'id' in lines[i]:  # Look for the next trial
            i += 1
        continue

    moves = []
    while i < len(lines) and 'id' in lines[i]:  # Parse moves within the trial
        move = lines[i]

        # Handle missing `eld_action`
        if move['actor'] == 'hel' and move.get('eld_action', '') == '':
            move['eld_action'] = 0

        # Handle missing `action` for helper
        if move['actor'] == 'hel' and move.get('action', '') == '':
            if i > 0 and lines[i - 1]['actor'] == 'hel' and 'action' in lines[i - 1]:
                move['action'] = lines[i - 1]['action']
            elif i + 1 < len(lines) and lines[i + 1]['actor'] == 'hel' and 'action' in lines[i + 1]:
                move['action'] = lines[i + 1]['action']
            else:
                move['action'] = 6  # Default to 6

        # Handle missing `eld_sees_o`
        if 'eld_sees_o' not in move or move['eld_sees_o'] == '':
            if moves:
                move['eld_sees_o'] = moves[-1]['eld_sees_o']
            else:
                move['eld_sees_o'] = init_sees

        moves.append({
            "actor": move['actor'],
            "utterance": move.get('utt', ''),
            "da": move.get('da', ''),
            "pt": move.get('pt', 0),
            "ho": move.get('ho', 0),
            "eldL": move.get('pt_target', 0),
            "eldOtGiven": move.get('prev_ot_given', 0),
            "eldLGiven": move.get('prev_l_given', 0),
            "helOtBelief": move.get('prev_hel_ot_belief', 0),
            "helLBelief": move.get('prev_hel_l_belief', 0),
            "helOBelief": move.get('prev_hel_o_belief', 0),
            "eldAction": move.get('eld_action', 0),
            "eldSeesO": move.get('eld_sees_o', 0)
        })

        i += 1

    test_corpus.append({
        "trial_id": trial_id,
        "ot": ot,
        "o": o,
        "init_action": init_action,
        "moves": moves
    })

    i += 1  # Move to the next trial


In [ ]:
test_condensed_corpus = []  # Combine consecutive moves by the same actor if state doesn't change

for trial in test_corpus:
    condensed_moves = []  # Initialize condensed moves list

    for move in trial["moves"]:
        if not condensed_moves:  # First move in trial
            condensed_moves.append(move)
        else:
            prev_move = condensed_moves[-1]

            # Check if actor and state are unchanged
            same_actor = prev_move["actor"] == move["actor"]
            same_state = (
                prev_move["pt"] == move["pt"]
                and prev_move["ho"] == move["ho"]
                and prev_move["eldL"] == move["eldL"]
                and prev_move["eldOtGiven"] == move["eldOtGiven"]
                and prev_move["eldLGiven"] == move["eldLGiven"]
                and prev_move["helOtBelief"] == move["helOtBelief"]
                and prev_move["helLBelief"] == move["helLBelief"]
                and prev_move["helOBelief"] == move["helOBelief"]
                and prev_move["eldAction"] == move["eldAction"]
                and prev_move["eldSeesO"] == move["eldSeesO"]
            )

            # Treat moves with pt/ho as separate
            if (move["pt"] or move["ho"]) and (prev_move["pt"] or prev_move["ho"]):
                condensed_moves.append(move)
                continue

            if same_actor and same_state:  # Condense moves
                # Combine utterances
                concatenated_utterance = " ".join([prev_move["utterance"], move["utterance"]]).strip()
                latest_da = (
                    prev_move["da"]
                    if prev_move["da"] and (not move["da"] or move["da"] == "Explain")
                    else move["da"]
                )
                latest_pt = prev_move["pt"] if prev_move["pt"] and not move["pt"] else move["pt"]
                latest_ho = prev_move["ho"] if prev_move["ho"] and not move["ho"] else move["ho"]

                # Update previous move
                condensed_moves[-1] = {
                    "actor": move["actor"],
                    "utterance": concatenated_utterance,
                    "da": latest_da,
                    "pt": latest_pt,
                    "ho": latest_ho,
                    "eldL": move["eldL"],
                    "eldOtGiven": move["eldOtGiven"],
                    "eldLGiven": move["eldLGiven"],
                    "helOtBelief": move["helOtBelief"],
                    "helLBelief": move["helLBelief"],
                    "helOBelief": move["helOBelief"],
                    "eldAction": move["eldAction"],
                    "eldSeesO": move["eldSeesO"],
                }
            else:
                condensed_moves.append(move)

    # Add condensed trial to the corpus
    test_condensed_corpus.append(
        {
            "trial_id": trial["trial_id"],
            "ot": trial["ot"],
            "o": trial["o"],
            "init_action": trial["init_action"],
            "moves": condensed_moves,
        }
    )


In [ ]:
test_condensed_corpus[0]

{'trial_id': '2010-11-11-002-ep04-104.txt|10',
 'ot': 'fruit',
 'o': 'grapes',
 'init_action': 0,
 'moves': [{'actor': 'hel',
   'utterance': 'action',
   'da': '',
   'pt': 0,
   'ho': 2,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 1,
   'helLBelief': 0,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 0},
  {'actor': 'hel',
   'utterance': 'okay',
   'da': 'Acknowledge',
   'pt': 0,
   'ho': 0,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 1,
   'helLBelief': 1,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 0},
  {'actor': 'hel',
   'utterance': 'here is the napkins',
   'da': 'State-y',
   'pt': 0,
   'ho': 1,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'helOtBelief': 1,
   'helLBelief': 1,
   'helOBelief': 0,
   'eldAction': 0,
   'eldSeesO': 0},
  {'actor': 'hel',
   'utterance': 'is this glass fine',
   'da': 'Query-yn',
   'pt': 0,
   'ho': 1,
   'eldL': 0,
   'eldOtGiven': 1,
   'eldLGiven': 1,
   'hel

In [ ]:
# Extract data points with the correct fields including prev_eld_da as specified
train_helper_data= []
for trial in train_condensed_corpus:
    for i, move in enumerate(trial["moves"]):
        if move[0] != "hel":
            continue

        # Find previous helper move and elder utterance
        prev_hel_move = None
        prev_eld_utt = ""
        prev_eld_da = ""
        for j in range(i-1, -1, -1):
            if trial["moves"][j][0] == 'hel' and prev_hel_move is None:
                prev_hel_move = trial["moves"][j]
            elif trial["moves"][j][0] == 'eld':
                prev_eld_utt = trial["moves"][j][1]
                prev_eld_da = trial["moves"][j][2]
                break

        if prev_hel_move is None:
            continue

        datapoint = {
            "prev_utt": re.sub("\s\s+", " ", re.sub("[.?!,]|xxx", "", prev_hel_move[1].lower() if prev_hel_move[1] else "action")),
            "current_utt": re.sub("\s\s+", " ", re.sub("[.?!,]|xxx", "", move[1].lower() if move[1] else "action")),
            "elder_utt": re.sub("\s\s+", " ", re.sub("[.?!,]|xxx", "", prev_eld_utt.lower() if prev_eld_utt else "action")),
            "prev_hel_belief": {
                "ot_belief": int(prev_hel_move[8]),
                "l_belief": int(prev_hel_move[9]),
                "o_belief": int(prev_hel_move[10])
            },
            "hel_belief": {
                "ot_belief": int(move[8]),
                "l_belief": int(move[9]),
                "o_belief": int(move[10])
            },
            "prev_hel_action": int(prev_hel_move[11]),
            "da": move[2],
            "hel_action": int(move[11]),
            "prev_eld_da": prev_eld_da
        }
        train_helper_data.append(datapoint)

# Save to JSON
json_file_path = 'train_aug_helper_data.json'
with open(json_file_path, 'w') as json_file:
    json.dump(train_helper_data, json_file, indent=4)

print("Data saved successfully!")

KeyError: 0

In [ ]:
import re
import json

train_helper_data = []

for trial in train_condensed_corpus:
    for i, move in enumerate(trial["moves"]):
        if move["actor"] != "hel":
            continue

        # Find previous helper move and elder utterance
        prev_hel_move = None
        prev_eld_utt = ""
        prev_eld_da = ""
        for j in range(i - 1, -1, -1):
            if trial["moves"][j]["actor"] == "hel" and prev_hel_move is None:
                prev_hel_move = trial["moves"][j]
            elif trial["moves"][j]["actor"] == "eld":
                prev_eld_utt = trial["moves"][j].get("utterance", "")
                prev_eld_da = trial["moves"][j].get("da", "")
                break

        if prev_hel_move is None:
            continue

        datapoint = {
            "prev_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_hel_move["utterance"].lower() if prev_hel_move["utterance"] else "action")
            ),
            "current_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", move["utterance"].lower() if move["utterance"] else "action")
            ),
            "elder_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_eld_utt.lower() if prev_eld_utt else "action")
            ),
            "prev_hel_belief": {
                "ot_belief": int(prev_hel_move["helOtBelief"]),
                "l_belief": int(prev_hel_move["helLBelief"]),
                "o_belief": int(prev_hel_move["helOBelief"]),
            },
            "hel_belief": {
                "ot_belief": int(move["helOtBelief"]),
                "l_belief": int(move["helLBelief"]),
                "o_belief": int(move["helOBelief"]),
            },
            "prev_hel_action": int(prev_hel_move["eldAction"]),
            "da": move["da"],
            "hel_action": int(move["eldAction"]),
            "prev_eld_da": prev_eld_da,
        }
        train_helper_data.append(datapoint)

# Save to JSON
json_file_path = "train_aug_helper_data.json"
with open(json_file_path, "w") as json_file:
    json.dump(train_helper_data, json_file, indent=4)

print("Data saved successfully!")


Data saved successfully!


In [ ]:
import re
import json

valid_helper_data = []

for trial in valid_condensed_corpus:
    for i, move in enumerate(trial["moves"]):
        if move["actor"] != "hel":
            continue

        # Find previous helper move and elder utterance
        prev_hel_move = None
        prev_eld_utt = ""
        prev_eld_da = ""
        for j in range(i - 1, -1, -1):
            if trial["moves"][j]["actor"] == "hel" and prev_hel_move is None:
                prev_hel_move = trial["moves"][j]
            elif trial["moves"][j]["actor"] == "eld":
                prev_eld_utt = trial["moves"][j].get("utterance", "")
                prev_eld_da = trial["moves"][j].get("da", "")
                break

        if prev_hel_move is None:
            continue

        datapoint = {
            "prev_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_hel_move["utterance"].lower() if prev_hel_move["utterance"] else "action")
            ),
            "current_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", move["utterance"].lower() if move["utterance"] else "action")
            ),
            "elder_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_eld_utt.lower() if prev_eld_utt else "action")
            ),
            "prev_hel_belief": {
                "ot_belief": int(prev_hel_move["helOtBelief"]),
                "l_belief": int(prev_hel_move["helLBelief"]),
                "o_belief": int(prev_hel_move["helOBelief"]),
            },
            "hel_belief": {
                "ot_belief": int(move["helOtBelief"]),
                "l_belief": int(move["helLBelief"]),
                "o_belief": int(move["helOBelief"]),
            },
            "prev_hel_action": int(prev_hel_move["eldAction"]),
            "da": move["da"],
            "hel_action": int(move["eldAction"]),
            "prev_eld_da": prev_eld_da,
        }
        valid_helper_data.append(datapoint)

# Save to JSON
json_file_path = "valid_aug_helper_data.json"
with open(json_file_path, "w") as json_file:
    json.dump(valid_helper_data, json_file, indent=4)

print("Data saved successfully!")


Data saved successfully!


In [ ]:
import re
import json

test_helper_data = []

for trial in test_condensed_corpus:
    for i, move in enumerate(trial["moves"]):
        if move["actor"] != "hel":
            continue

        # Find previous helper move and elder utterance
        prev_hel_move = None
        prev_eld_utt = ""
        prev_eld_da = ""
        for j in range(i - 1, -1, -1):
            if trial["moves"][j]["actor"] == "hel" and prev_hel_move is None:
                prev_hel_move = trial["moves"][j]
            elif trial["moves"][j]["actor"] == "eld":
                prev_eld_utt = trial["moves"][j].get("utterance", "")
                prev_eld_da = trial["moves"][j].get("da", "")
                break

        if prev_hel_move is None:
            continue

        datapoint = {
            "prev_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_hel_move["utterance"].lower() if prev_hel_move["utterance"] else "action")
            ),
            "current_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", move["utterance"].lower() if move["utterance"] else "action")
            ),
            "elder_utt": re.sub(
                r"\s\s+", " ",
                re.sub(r"[.?!,]|xxx", "", prev_eld_utt.lower() if prev_eld_utt else "action")
            ),
            "prev_hel_belief": {
                "ot_belief": int(prev_hel_move["helOtBelief"]),
                "l_belief": int(prev_hel_move["helLBelief"]),
                "o_belief": int(prev_hel_move["helOBelief"]),
            },
            "hel_belief": {
                "ot_belief": int(move["helOtBelief"]),
                "l_belief": int(move["helLBelief"]),
                "o_belief": int(move["helOBelief"]),
            },
            "prev_hel_action": int(prev_hel_move["eldAction"]),
            "da": move["da"],
            "hel_action": int(move["eldAction"]),
            "prev_eld_da": prev_eld_da,
        }
        test_helper_data.append(datapoint)

# Save to JSON
json_file_path = "test_aug_helper_data.json"
with open(json_file_path, "w") as json_file:
    json.dump(test_helper_data, json_file, indent=4)

print("Data saved successfully!")


Data saved successfully!


In [ ]:
# Combine datasets
combined_data = train_helper_data + valid_helper_data + test_helper_data
print(len(combined_data))
json_file_path = "augmented_helper_data.json"
with open(json_file_path, "w") as json_file:
    json.dump(combined_data, json_file, indent=4)

1823


In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import torch
from transformers import BertTokenizer, BertModel
from sklearn.utils import resample

In [ ]:
random.seed(42)
# Load the data
with open('augmented_helper_data.json', 'r') as json_file:
    data = json.load(json_file)

# Convert data to DataFrame
df = pd.DataFrame(data)

# Initialize BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to get BERT embeddings
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

# Get BERT embeddings for utterances
df['eld_utt_emb'] = df['elder_utt'].apply(get_bert_embeddings)
df['prev_utt_emb'] = df['prev_utt'].apply(get_bert_embeddings)

# Encode the dialogue acts and actions
label_encoder_da = LabelEncoder()
label_encoder_da.fit(df['da'].tolist() + df['prev_eld_da'].tolist())
df['da'] = label_encoder_da.transform(df['da'])
df['eld_da'] = label_encoder_da.transform(df['prev_eld_da'])

label_encoder_action = LabelEncoder()
label_encoder_action.fit(df['prev_hel_action'].tolist() + df['hel_action'].tolist())
df['prev_hel_action'] = label_encoder_action.transform(df['prev_hel_action'])
df['hel_action'] = label_encoder_action.transform(df['hel_action'])


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
# Combine features
X = np.concatenate([
    np.vstack(df['eld_utt_emb']),
    np.vstack(df['prev_utt_emb']),
    df[['eld_da']].values,
    np.vstack(df['prev_hel_belief'].apply(lambda x: list(x.values())).values),
], axis=1)

y_da = df['da'].values
y_action = df['hel_action'].values

# Function to split data manually based on the given class distribution
def manual_split(X, y_da, y_action):
    train_indices = []
    val_indices = []
    test_indices = []

    # Split y_action
    for cls in np.unique(y_action):
        indices = np.where(y_action == cls)[0]
        np.random.shuffle(indices)
        if cls in [1, 3]:  # Classes with only 4 samples
            if len(indices) >= 4:
                train_indices.extend(indices[:2])
                val_indices.append(indices[2])
                test_indices.append(indices[3])
            elif len(indices) == 3:
                train_indices.extend(indices[:1])
                val_indices.append(indices[1])
                test_indices.append(indices[2])
            elif len(indices) == 2:
                train_indices.extend(indices[:1])
                val_indices.append(indices[1])
            elif len(indices) == 1:
                train_indices.append(indices[0])
        else:
            split1 = int(0.7 * len(indices))
            split2 = int(0.85 * len(indices))
            train_indices.extend(indices[:split1])
            val_indices.extend(indices[split1:split2])
            test_indices.extend(indices[split2:])

    # Split y_da
    for cls in np.unique(y_da):
        indices = np.where(y_da == cls)[0]
        np.random.shuffle(indices)
        if cls in [1]:  # Classes with only 3 samples
            if len(indices) >= 3:
                train_indices.append(indices[0])
                val_indices.append(indices[1])
                test_indices.append(indices[2])
            elif len(indices) == 2:
                train_indices.extend(indices[:1])
                val_indices.append(indices[1])
            elif len(indices) == 1:
                train_indices.append(indices[0])
        elif cls in [2, 3, 4, 9]:  # Classes with 4 samples
            if len(indices) >= 4:
                train_indices.extend(indices[:2])
                val_indices.append(indices[2])
                test_indices.append(indices[3])
            elif len(indices) == 3:
                train_indices.extend(indices[:1])
                val_indices.append(indices[1])
                test_indices.append(indices[2])
            elif len(indices) == 2:
                train_indices.extend(indices[:1])
                val_indices.append(indices[1])
            elif len(indices) == 1:
                train_indices.append(indices[0])
        else:
            split1 = int(0.7 * len(indices))
            split2 = int(0.85 * len(indices))
            train_indices.extend(indices[:split1])
            val_indices.extend(indices[split1:split2])
            test_indices.extend(indices[split2:])

    # Remove duplicates and ensure indices are unique
    train_indices = list(set(train_indices))
    val_indices = list(set(val_indices))
    test_indices = list(set(test_indices))

    # Split the data
    X_train, y_da_train, y_action_train = X[train_indices], y_da[train_indices], y_action[train_indices]
    X_val, y_da_val, y_action_val = X[val_indices], y_da[val_indices], y_action[val_indices]
    X_test, y_da_test, y_action_test = X[test_indices], y_da[test_indices], y_action[test_indices]

    return X_train, X_val, X_test, y_da_train, y_da_val, y_da_test, y_action_train, y_action_val, y_action_test

# Manually split the data
X_train, X_val, X_test, y_da_train, y_da_val, y_da_test, y_action_train, y_action_val, y_action_test = manual_split(X, y_da, y_action)

# Function to ensure all classes are in the training set
def ensure_all_classes(X_train, y_train, expected_classes):
    unique_classes = np.unique(y_train)
    missing_classes = set(expected_classes) - set(unique_classes)

    for cls in missing_classes:
        # Find an example from the original dataset
        example_idx = np.where(y_da == cls)[0]
        if len(example_idx) > 0:
            example_idx = example_idx[0]
            X_train = np.vstack([X_train, X[example_idx]])
            y_train = np.append(y_train, cls)

    return X_train, y_train

# Ensure all expected classes are in the training set for dialogue acts
expected_classes_da = np.arange(9)  # Assuming classes 0 to 8
X_train, y_da_train = ensure_all_classes(X_train, y_da_train, expected_classes_da)

# Upsample the training data
def upsample_training_data(X_train, y_da_train, y_action_train):
    # Concatenate the data to simplify the upsampling process
    train_data = np.concatenate([X_train, y_da_train.reshape(-1, 1), y_action_train.reshape(-1, 1)], axis=1)
    df_train = pd.DataFrame(train_data)

    # Upsample y_da_train
    for cls in np.unique(y_da_train):
        class_data = df_train[df_train.iloc[:, -2] == cls]
        if len(class_data) < 3:
            upsampled_class_data = resample(class_data, replace=True, n_samples=3, random_state=42)
            df_train = pd.concat([df_train, upsampled_class_data], axis=0)

    # Upsample y_action_train
    for cls in np.unique(y_action_train):
        class_data = df_train[df_train.iloc[:, -1] == cls]
        if len(class_data) < 3:
            upsampled_class_data = resample(class_data, replace=True, n_samples=3, random_state=42)
            df_train = pd.concat([df_train, upsampled_class_data], axis=0)

    # Separate the features and labels after upsampling
    X_train_upsampled = df_train.iloc[:, :-2].values
    y_da_train_upsampled = df_train.iloc[:, -2].values.astype(int)  # Convert to integer
    y_action_train_upsampled = df_train.iloc[:, -1].values.astype(int)  # Convert to integer

    return X_train_upsampled, y_da_train_upsampled, y_action_train_upsampled

X_train, y_da_train, y_action_train = upsample_training_data(X_train, y_da_train, y_action_train)

# Remap the class labels to be continuous integers
label_encoder_da = LabelEncoder()
y_da_train = label_encoder_da.fit_transform(y_da_train)
y_da_val = label_encoder_da.transform(y_da_val)
y_da_test = label_encoder_da.transform(y_da_test)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import random

random.seed(42)
# Calculate class weights for imbalanced classes
class_weights_da = dict(zip(np.unique(y_da_train), 1.0 / np.bincount(y_da_train)))
class_weights_action = dict(zip(np.unique(y_action_train), 1.0 / np.bincount(y_action_train)))

# Generate sample weights for each instance in the training data
sample_weights_da = np.array([class_weights_da[label] for label in y_da_train])
sample_weights_action = np.array([class_weights_action[label] for label in y_action_train])

# Define parameter grid for tuning Logistic Regression
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': np.logspace(-4, 4, 20),
    'solver': ['liblinear', 'lbfgs', 'saga'],  # solvers compatible with different penalties
    'max_iter': [100, 200, 500],
    'class_weight': ['balanced', None]
}

# Initialize Logistic Regression classifiers for Dialogue Acts and Actions
model_da = LogisticRegression(multi_class='multinomial', random_state=42)
model_action = LogisticRegression(multi_class='multinomial', random_state=42)

# Set up RandomizedSearchCV for Dialogue Acts classifier
random_search_da = RandomizedSearchCV(
    estimator=model_da,
    param_distributions=param_grid,
    n_iter=50,  # Number of parameter combinations to try
    scoring='f1_weighted',  # Optimize for F1 score
    cv=3,  # 3-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Set up RandomizedSearchCV for Actions classifier
random_search_action = RandomizedSearchCV(
    estimator=model_action,
    param_distributions=param_grid,
    n_iter=50,  # Number of parameter combinations to try
    scoring='f1_weighted',  # Optimize for F1 score
    cv=3,  # 3-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the randomized search model for Dialogue Acts
random_search_da.fit(X_train, y_da_train, sample_weight=sample_weights_da)
best_params_da = random_search_da.best_params_
best_model_da = random_search_da.best_estimator_

# Fit the randomized search model for Actions
random_search_action.fit(X_train, y_action_train, sample_weight=sample_weights_action)
best_params_action = random_search_action.best_params_
best_model_action = random_search_action.best_estimator_

print("Best parameters for Dialogue Acts:", best_params_da)
print("Best parameters for Actions:", best_params_action)

# Evaluate the best models on the test set
y_da_pred = best_model_da.predict(X_test)
y_action_pred = best_model_action.predict(X_test)

accuracy_da = accuracy_score(y_da_test, y_da_pred)
f1_score_da = f1_score(y_da_test, y_da_pred, average='weighted')
accuracy_action = accuracy_score(y_action_test, y_action_pred)
f1_score_action = f1_score(y_action_test, y_action_pred, average='weighted')

print(f"Best Accuracy for Dialogue Acts: {accuracy_da}")
print(f"Best F1-score for Dialogue Acts: {f1_score_da}")
print(f"Best Accuracy for Actions: {accuracy_action}")
print(f"Best F1-score for Actions: {f1_score_action}")


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
117 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
33 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py", line 1267, in fit
    multi_class = _check_multi_class(multi_class, solver, len(self.classes

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
117 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
33 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py", line 1267, in fit
    multi_class = _check_multi_class(multi_class, solver, len(self.classes

Best parameters for Dialogue Acts: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 200, 'class_weight': None, 'C': 1.623776739188721}
Best parameters for Actions: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 100, 'class_weight': None, 'C': 545.5594781168514}
Best Accuracy for Dialogue Acts: 0.3139240506329114
Best F1-score for Dialogue Acts: 0.3264217166552677
Best Accuracy for Actions: 0.34177215189873417
Best F1-score for Actions: 0.3586246709289415


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
print(f"Best Accuracy for Dialogue Acts: {accuracy_da}")
print(f"Best F1-score for Dialogue Acts: {f1_score_da}")
print(f"Best Accuracy for Actions: {accuracy_action}")
print(f"Best F1-score for Actions: {f1_score_action}")

Best Accuracy for Dialogue Acts: 0.3139240506329114
Best F1-score for Dialogue Acts: 0.3264217166552677
Best Accuracy for Actions: 0.34177215189873417
Best F1-score for Actions: 0.3586246709289415


In [ ]:
#LR models that are fine tuned + class weights
import joblib

# Save the Dialogue Acts model
joblib.dump(best_model_da, 'aug_best_lr_model_da.json')

# Save the Actions model
joblib.dump(best_model_action, 'aug_best_lr_model_action.json')

print("Models saved successfully!")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import random

random.seed(42)

# Define parameter grid for tuning Logistic Regression
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': np.logspace(-4, 4, 20),
    'solver': ['liblinear', 'lbfgs', 'saga'],  # solvers compatible with different penalties
    'max_iter': [100, 200, 500]
}

# Initialize Logistic Regression classifiers for Dialogue Acts and Actions
model_da = LogisticRegression(multi_class='multinomial', random_state=42)
model_action = LogisticRegression(multi_class='multinomial', random_state=42)

# Set up RandomizedSearchCV for Dialogue Acts classifier
random_search_da = RandomizedSearchCV(
    estimator=model_da,
    param_distributions=param_grid,
    n_iter=50,  # Number of parameter combinations to try
    scoring='f1_weighted',  # Optimize for F1 score
    cv=3,  # 3-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Set up RandomizedSearchCV for Actions classifier
random_search_action = RandomizedSearchCV(
    estimator=model_action,
    param_distributions=param_grid,
    n_iter=50,  # Number of parameter combinations to try
    scoring='f1_weighted',  # Optimize for F1 score
    cv=3,  # 3-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the randomized search model for Dialogue Acts
random_search_da.fit(X_train, y_da_train)
best_params_da = random_search_da.best_params_
best_model_da = random_search_da.best_estimator_

# Fit the randomized search model for Actions
random_search_action.fit(X_train, y_action_train)
best_params_action = random_search_action.best_params_
best_model_action = random_search_action.best_estimator_

print("Best parameters for Dialogue Acts:", best_params_da)
print("Best parameters for Actions:", best_params_action)

# Evaluate the best models on the test set
y_da_pred = best_model_da.predict(X_test)
y_action_pred = best_model_action.predict(X_test)

accuracy_da = accuracy_score(y_da_test, y_da_pred)
f1_score_da = f1_score(y_da_test, y_da_pred, average='weighted')
accuracy_action = accuracy_score(y_action_test, y_action_pred)
f1_score_action = f1_score(y_action_test, y_action_pred, average='weighted')

print(f"Best Accuracy for Dialogue Acts: {accuracy_da}")
print(f"Best F1-score for Dialogue Acts: {f1_score_da}")
print(f"Best Accuracy for Actions: {accuracy_action}")
print(f"Best F1-score for Actions: {f1_score_action}")


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
108 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py", line 1194, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
  File "/

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
108 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py", line 1194, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
  File "/

Best parameters for Dialogue Acts: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 500, 'C': 10000.0}
Best parameters for Actions: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 200, 'C': 0.615848211066026}
Best Accuracy for Dialogue Acts: 0.4936708860759494
Best F1-score for Dialogue Acts: 0.47919844580707044
Best Accuracy for Actions: 0.4151898734177215
Best F1-score for Actions: 0.4029608602708546


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
print(f"Best Accuracy for Dialogue Acts: {accuracy_da}")
print(f"Best F1-score for Dialogue Acts: {f1_score_da}")
print(f"Best Accuracy for Actions: {accuracy_action}")
print(f"Best F1-score for Actions: {f1_score_action}")

Best Accuracy for Dialogue Acts: 0.4936708860759494
Best F1-score for Dialogue Acts: 0.47919844580707044
Best Accuracy for Actions: 0.4151898734177215
Best F1-score for Actions: 0.4029608602708546
